# LIME Local Explanations for TLDC

This notebook produces a local perturbation-based explanation for a single document prediction. LIME feature weights depend on the perturbation distribution; word deletion can distort Telugu morphology or legal identifiers. Review each highlighted passage against the source page.

Execution is opt-in and requires an approved dataset plus a saved model at `models/classical_ml/document_classifier.joblib`. Explanations are not claims about legal correctness.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL_PATH = ROOT / "models/classical_ml/document_classifier.joblib"
TEST_PATH = ROOT / "data/test/tldc_test.jsonl"
SEGMENTS_PATH = ROOT / "annotations/document_segmentation/segmented_documents.jsonl"
OUTPUT_PATH = ROOT / "xai/lime/lime_results.jsonl"
RUN_LIME = False


def read_jsonl(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as source:
        return [json.loads(line) for line in source if line.strip()]


test_rows = [row for row in read_jsonl(TEST_PATH) if row.get("task") == "document_classification"]
if not RUN_LIME:
    print("LIME is disabled; set RUN_LIME=True only after a saved model and reviewed test example are available.")
elif not MODEL_PATH.exists() or not test_rows:
    print("Skip LIME: saved model and held-out reviewed examples are required.")
else:
    import joblib
    from src.explainability.lime_explainer import explain_text
    from src.explainability.evidence_linker import link_attributions

    model = joblib.load(MODEL_PATH)
    source_documents = {row["document_id"]: row for row in read_jsonl(SEGMENTS_PATH)}
    example = test_rows[0]
    source_document = source_documents.get(example["document_id"], {})
    explanation = explain_text(model, example["text"], model_version="tldc_baseline_v1")
    record = link_attributions(
        example["document_id"], example["text"], source_document.get("sections", []),
        explanation["attributions"], explanation["predicted_label"], explanation["model_version"],
        explanation["method"], score=explanation["score"],
    )
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_PATH.open("a", encoding="utf-8") as output:
        output.write(json.dumps(record, ensure_ascii=False) + "\n")
    record

LIME is disabled; set RUN_LIME=True only after a saved model and reviewed test example are available.


In [2]:
# Compare one local explanation per language group when both Telugu and mixed examples exist.
if RUN_LIME and "model" in globals() and test_rows:
    from src.explainability.lime_explainer import explain_text

    for language in ("Telugu", "Mixed"):
        language_rows = [row for row in test_rows if row.get("language") == language]
        if language_rows:
            local = explain_text(model, language_rows[0]["text"], model_version="tldc_baseline_v1")
            print(language, local["predicted_label"], local["attributions"][:10])
        else:
            print(language, "slice unavailable")
else:
    print("Telugu/mixed LIME comparison awaits a saved model and eligible examples.")

Telugu/mixed LIME comparison awaits a saved model and eligible examples.
